# Sparse MoE Transformer: Colab runner
Runtime → Change runtime type → **T4 GPU**. Upload `sparse-moe-transformer.zip` to **My Drive** first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive
!unzip -q -o sparse-moe-transformer.zip
%cd sparse-moe-transformer
!pip install -q tiktoken datasets
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m tests.test_moe

In [ ]:
# Quick sanity run (~ a few minutes)
!python train.py --name dense_s0 --ffn dense --d_ff 1024 --steps 2000
!python train.py --name top2m_e8_s0 --ffn moe --top_k 2 --n_experts 8 --d_ff 512 --steps 2000

In [ ]:
# Full sweep (long). Save results to Drive if your session might disconnect.
!SEEDS="0" STEPS=2000 bash run_experiments.sh

In [ ]:
!python profile_moe.py
!python analyze.py

In [ ]:
import os
from IPython.display import Markdown, Image, display
display(Markdown(open('results/summary.md').read()))
for p in ['results/ppl_vs_experts.png','results/latency.png','results/expert_load.png']:
    if os.path.exists(p):
        display(Image(p))
    else:
        print('missing (no training runs yet?):', p)

In [ ]:
!zip -qr results.zip results && echo 'download results.zip from the Files panel'